# Import Libraries and Data Importing


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
df = pd.read_csv("/workspaces/Reserving-using-Agentic-AI/data/comauto_pos.csv")

# Data Exploration

In [9]:
df.head()

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurLoss_C,CumPaidLoss_C,BulkLoss_C,EarnedPremDIR_C,EarnedPremCeded_C,EarnedPremNet_C,Single,PostedReserve97_C
0,266,Public Underwriters Grp,1988,1988,1,0,0,0,0,0,0,0,932
1,266,Public Underwriters Grp,1988,1989,2,0,0,0,0,0,0,0,932
2,266,Public Underwriters Grp,1988,1990,3,0,0,0,0,0,0,0,932
3,266,Public Underwriters Grp,1988,1991,4,0,0,0,0,0,0,0,932
4,266,Public Underwriters Grp,1988,1992,5,0,0,0,0,0,0,0,932


In [10]:
df['GRCODE'].unique()

array([  266,   337,   353,   388,   460,   620,   655,   671,   715,
         833,   965,  1066,  1090,  1279,  1538,  1716,  1767,  2003,
        2135,  2143,  2208,  2569,  2623,  2712,  3131,  3240,  3492,
        4839,  5185,  5320,  5690,  5940,  6408,  6459,  6777,  6807,
        6947,  7080,  8079,  8281,  8427,  8559,  8672,  9466, 10019,
       10022, 10048, 10074, 10100, 10308, 10561, 10790, 10859, 10894,
       11037, 11118, 11126, 11150, 11231, 11460, 12866, 13420, 13439,
       13501, 13528, 13587, 13641, 13889, 13943, 14044, 14176, 14257,
       14311, 14320, 14370, 14508, 14974, 15024, 15199, 15407, 15792,
       15911, 15997, 16411, 16748, 17299, 17884, 18163, 18309, 18380,
       18538, 18686, 18767, 18791, 19020, 19780, 20451, 20690, 21172,
       21270, 22390, 23574, 23663, 25275, 25950, 26077, 26433, 26797,
       26905, 27022, 27065, 27499, 27955, 27980, 28258, 28436, 28535,
       28550, 28886, 29297, 29378, 29440, 31550, 31810, 32301, 32514,
       32670, 32743,

In [11]:
df_company = df[df['GRCODE'] == 13587]

# Functions

In [ ]:
def build_triangle( df, index_col, column_col, value_col, aggfunc="sum"):
    """
    Builds a triangle from a long-form DataFrame.
    """
    return df.pivot_table(
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc=aggfunc
    )

def incremental_to_cumulative(triangle: pd.DataFrame) -> pd.DataFrame:
    """
    Convert incremental loss triangle to cumulative loss triangle.
    Also ensures columns are typed as integers.
    """
    cumulative = triangle.cumsum(axis=1)
    cumulative.columns = cumulative.columns.astype(int)
    return cumulative

def calculate_age_to_age_factors(cumulative_triangle):
    """
    Calculate individual age-to-age factors for every accident year.
    Parameters
    ----------
    cumulative_triangle : pd.DataFrame
        Cumulative loss triangle
    Returns
    -------
    pd.DataFrame
        Age-to-age factor matrix
    """

    factors = pd.DataFrame(index=cumulative_triangle.index)
    for col in range(cumulative_triangle.shape[1] - 1):
        current = cumulative_triangle.iloc[:, col]
        nxt = cumulative_triangle.iloc[:, col + 1]
        factors[f"{col+1}->{col+2}"] = nxt / current
    return factors

def plot_age_to_age(age_to_age_triangle):

    for col in age_to_age_triangle.columns:

        plt.figure(figsize=(8,4))

        plt.plot(
            age_to_age_triangle.index,
            age_to_age_triangle[col],
            marker="o"
        )

        plt.title(f"Age-to-Age Factor {col}")
        plt.xlabel("Accident Year")
        plt.ylabel("Factor")
        plt.grid(True)

        plt.show()

def calculate_average_factors(age_to_age_triangle):

    results = {}

    for col in age_to_age_triangle.columns:

        factors = age_to_age_triangle[col].dropna()

        simple = factors.mean()

        geometric = np.prod(factors) ** (1 / len(factors))

        if len(factors) >= 3:
            medial = (
                factors.sort_values()
                .iloc[1:-1]
                .mean()
            )
        else:
            medial = simple

        results[col] = {
            "Simple": simple,
            "Geometric": geometric,
            "Medial": medial
        }

    return pd.DataFrame(results)

def plot_average_factors(avg_factors):

    plt.figure(figsize=(10,5))

    for avg_type in avg_factors.index:

        plt.plot(
            avg_factors.columns,
            avg_factors.loc[avg_type],
            marker='o',
            label=avg_type
        )

    plt.xlabel("Development Period")
    plt.ylabel("Factor")
    plt.title("Average Development Factors")
    plt.legend()
    plt.grid(True)

    plt.show()

def select_ldfs(avg_factors):

    print("Select LDF Method:")
    print("1. Simple Average")
    print("2. Geometric Average")
    print("3. Medial Average")

    choice = input("Enter choice (1-3): ")

    method_map = {
        "1": "Simple",
        "2": "Geometric",
        "3": "Medial"
    }

    selected_ldfs = avg_factors.loc[method_map[choice]].copy()

    tail_input = input(
        "Enter Tail Factor (press Enter for 1.0): "
    )

    tail_factor = (
        float(tail_input)
        if tail_input.strip()
        else 1.0
    )

    selected_ldfs["Tail"] = tail_factor

    return selected_ldfs

def calculate_cdfs(selected_ldfs):

    cdfs = []

    factors = selected_ldfs.values

    for i in range(len(factors)):
        cdfs.append(np.prod(factors[i:]))

    return pd.Series(cdfs)

def project_ultimates(triangle, cdfs):

    cdfs = cdfs.copy()
    cdfs.index = triangle.columns

    results = []

    for ay in triangle.index:

        row = triangle.loc[ay].dropna()

        latest_age = row.index[-1]
        latest_value = row.iloc[-1]

        cdf = cdfs.loc[latest_age]

        ultimate = latest_value * cdf

        results.append({
            "AccidentYear": ay,
            "LatestAge": latest_age,
            "LatestValue": latest_value,
            "CDF": cdf,
            "Ultimate": ultimate
        })

    return pd.DataFrame(results)

def calculate_unpaid_claim_estimate(ultimate_df):

    reserve_df = ultimate_df.copy()
    reserve_df["TotalReserveEstimate"] = reserve_df["Ultimate"] - reserve_df["LatestValue"]

    return reserve_df

def calculate_total_reserve(reserve_df):

    total_reserve = reserve_df["TotalReserveEstimate"].sum()
    return total_reserve

def calculate_case_outstanding(paid_ultimate_df,incurred_ultimate_df):
    
    result = pd.DataFrame()
    
    result['AccidentYear'] = paid_ultimate_df["AccidentYear"]
    result["PaidLatest"] = paid_ultimate_df["LatestValue"]
    result["IncurredLatest"] = incurred_ultimate_df["LatestValue"]
    result["CaseOutstanding"] = (result["IncurredLatest"]-result["PaidLatest"])

    if (result["CaseOutstanding"] < 0).any():
        print("WARNING: Negative case reserves detected")

    return result

def calculate_ibnr(selected_ultimate_df,incurred_ultimate_df):

    result = pd.DataFrame()

    result["AccidentYear"] = selected_ultimate_df["AccidentYear"]
    result["SelectedUltimate"] = selected_ultimate_df["Ultimate"]
    result["CurrentIncurred"] = incurred_ultimate_df["LatestValue"]
    result["IBNR"] = result["SelectedUltimate"] - result["CurrentIncurred"]

    if (result["IBNR"] < 0).any():
        print("WARNING: Negative IBNR detected")

    return result

def reserve_summary( paid_ultimate_df, incurred_ultimate_df, selected_ultimate_df):
    # Build summary table
    result = pd.DataFrame()
    # Calculate reserve components
    case_df = calculate_case_outstanding(
        paid_ultimate_df,
        incurred_ultimate_df
    )

    ibnr_df = calculate_ibnr(
        selected_ultimate_df,
        incurred_ultimate_df
    )

    result["AccidentYear"] = selected_ultimate_df["AccidentYear"]
    result["Paid"] = paid_ultimate_df["LatestValue"]
    result["Incurred"] = incurred_ultimate_df["LatestValue"]
    result["SelectedUltimate"] = selected_ultimate_df["Ultimate"]
    result = result.merge(
        case_df[["AccidentYear", "CaseOutstanding"]],
        on="AccidentYear",
        how="left"
    )
    result.rename( columns={"CaseOutstanding": "CaseReserve"}, inplace=True)
    result = result.merge(
        ibnr_df[["AccidentYear", "IBNR"]],
        on="AccidentYear",
        how="left"
    )
    result["TotalReserve"] = result["SelectedUltimate"] - result["Paid"]
    result["Check"] = result["CaseReserve"] + result["IBNR"]
    result["Difference"] = result["TotalReserve"] - result["Check"]
    result['Status'] = np.where(
            np.isclose(result["Difference"], 0, atol=1),
            "Reconciles",
            "Different Ultimate Selected"
    )
    result["PaidCLUltimate"] = paid_ultimate_df["Ultimate"]
    result["IncurredCLUltimate"] = incurred_ultimate_df["Ultimate"]

    return result

def select_ultimate( method_results, selected_method ):
    return method_results[selected_method]["ultimate_df"]

# Data Preparation


In [13]:
paid = build_triangle(df_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='CumPaidLoss_C',aggfunc="sum" )
incurred = build_triangle(df_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='IncurLoss_C',aggfunc="sum" )

This is a fully squared triangle from CAS. So, this is being converted back to a triangular format

In [14]:
df_masked=df[df['DevelopmentYear']<=1997]
df_masked_company = df_masked[df_masked['GRCODE'] == 13587]
paid_masked = build_triangle(df_masked_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='CumPaidLoss_C',aggfunc="sum" )
incurred_masked = build_triangle(df_masked_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='IncurLoss_C',aggfunc="sum" )

In [15]:
#plot_age_to_age(paid_age_to_age_factors)
#plot_age_to_age(incurred_age_to_age_factors)

# One final function for full Chain Ladder Method

In [ ]:
def run_chain_ladder(triangle,method="Simple",tail_factor=1.0,show_plots=True):
    age_to_age = calculate_age_to_age_factors(triangle)

    if show_plots:
        plot_age_to_age(age_to_age)

    avg_ldfs = calculate_average_factors(age_to_age)
    print("\nAverage LDFs")
    display(avg_ldfs)

    if show_plots:
        plot_average_factors(avg_ldfs)

    # Select LDFs
    selected_ldfs = avg_ldfs.loc[method].copy()
    selected_ldfs["Tail"] = tail_factor
    print("\nSelected LDFs")
    display(selected_ldfs)

    cdfs = calculate_cdfs(selected_ldfs)
    print("\nCDFs")
    display(cdfs)

    # Ultimate Projection
    ultimate_df = project_ultimates( triangle, cdfs)
    print("\nUltimate Projection")
    display(ultimate_df)
    
    # Reserves
    reserve_df = calculate_unpaid_claim_estimate(ultimate_df)
    print("\nReserve Projection")
    display(reserve_df)
    # Total Reserve
    total_reserve = calculate_total_reserve(reserve_df)
    print(f"\nTotal Reserve = {total_reserve:,.2f}")
    return {
    "method": "Chain Ladder",
    "age_to_age": age_to_age,
    "avg_ldfs": avg_ldfs,
    "selected_ldfs": selected_ldfs,
    "cdfs": cdfs,
    "ultimate_df": ultimate_df,
    "reserve_df": reserve_df,
    "total_reserve": total_reserve
}

In [17]:
paid_results = run_chain_ladder(
    paid_masked,
    method="Simple",
    tail_factor=1.0,
    show_plots=False
)


Average LDFs


,1->2,2->3,3->4,4->5,5->6,6->7,7->8,8->9,9->10
Simple,2.010575,1.727571,1.187463,1.052004,1.219189,1.021429,1.0,1.0,1.0
Geometric,1.941390,1.508088,1.179350,1.051464,1.192573,1.020772,1.0,1.0,1.0
Medial,1.902136,1.475650,1.183436,1.055279,1.158564,1.000000,1.0,1.0,1.0



Selected LDFs


1->2     2.010575
2->3     1.727571
3->4     1.187463
4->5     1.052004
5->6     1.219189
6->7     1.021429
7->8     1.000000
8->9     1.000000
9->10    1.000000
Tail     1.000000
Name: Simple, dtype: float64


CDFs


...


Ultimate Projection


,AccidentYear,LatestAge,LatestValue,CDF,Ultimate
0,1988,10,86.0,1.000000,86.000000
1,1989,9,76.0,1.000000,76.000000
2,1990,8,128.0,1.000000,128.000000
3,1991,7,69.0,1.000000,69.000000
4,1992,6,175.0,1.021429,178.750000
5,1993,5,24.0,1.245315,29.887552
6,1994,4,143.0,1.310076,187.340931
7,1995,3,26.0,1.555667,40.447346
8,1996,2,37.0,2.687525,99.438423
9,1997,1,25.0,5.403469,135.086731



Reserve Projection


,AccidentYear,LatestAge,LatestValue,CDF,Ultimate,TotalReserveEstimate
0,1988,10,86.0,1.000000,86.000000,0.000000
1,1989,9,76.0,1.000000,76.000000,0.000000
2,1990,8,128.0,1.000000,128.000000,0.000000
3,1991,7,69.0,1.000000,69.000000,0.000000
4,1992,6,175.0,1.021429,178.750000,3.750000
5,1993,5,24.0,1.245315,29.887552,5.887552
6,1994,4,143.0,1.310076,187.340931,44.340931
7,1995,3,26.0,1.555667,40.447346,14.447346
8,1996,2,37.0,2.687525,99.438423,62.438423
9,1997,1,25.0,5.403469,135.086731,110.086731



Total Reserve = 240.95


In [18]:
incurred_results = run_chain_ladder(
    incurred_masked,
    method="Simple",
    tail_factor=1.0,
    show_plots=False
)


Average LDFs


,1->2,2->3,3->4,4->5,5->6,6->7,7->8,8->9,9->10
Simple,1.136027,1.153291,1.081218,1.022142,1.021385,0.984568,1.0,1.0,1.0
Geometric,1.100079,1.074797,1.066796,1.013388,1.017349,0.984197,1.0,1.0,1.0
Medial,1.044239,1.060993,1.056896,1.018625,0.986883,1.000000,1.0,1.0,1.0



Selected LDFs


1->2     1.136027
2->3     1.153291
3->4     1.081218
4->5     1.022142
5->6     1.021385
6->7     0.984568
7->8     1.000000
8->9     1.000000
9->10    1.000000
Tail     1.000000
Name: Simple, dtype: float64


CDFs


...


Ultimate Projection


,AccidentYear,LatestAge,LatestValue,CDF,Ultimate
0,1988,10,86.0,1.000000,86.000000
1,1989,9,76.0,1.000000,76.000000
2,1990,8,128.0,1.000000,128.000000
3,1991,7,69.0,1.000000,69.000000
4,1992,6,197.0,0.984568,193.959877
5,1993,5,24.0,1.005622,24.134938
6,1994,4,156.0,1.027889,160.350632
7,1995,3,69.0,1.111372,76.684681
8,1996,2,105.0,1.281736,134.582287
9,1997,1,134.0,1.456087,195.115606



Reserve Projection


,AccidentYear,LatestAge,LatestValue,CDF,Ultimate,TotalReserveEstimate
0,1988,10,86.0,1.000000,86.000000,0.000000
1,1989,9,76.0,1.000000,76.000000,0.000000
2,1990,8,128.0,1.000000,128.000000,0.000000
3,1991,7,69.0,1.000000,69.000000,0.000000
4,1992,6,197.0,0.984568,193.959877,-3.040123
5,1993,5,24.0,1.005622,24.134938,0.134938
6,1994,4,156.0,1.027889,160.350632,4.350632
7,1995,3,69.0,1.111372,76.684681,7.684681
8,1996,2,105.0,1.281736,134.582287,29.582287
9,1997,1,134.0,1.456087,195.115606,61.115606



Total Reserve = 99.83


In [19]:
paid_ultimate_df = paid_results["ultimate_df"]

incurred_ultimate_df = incurred_results["ultimate_df"]

selected_ultimate_df = paid_ultimate_df.copy() 

summary = reserve_summary(
    paid_ultimate_df,
    incurred_ultimate_df,
    selected_ultimate_df      # selected ultimate
)

display(summary)

,AccidentYear,Paid,Incurred,SelectedUltimate,CaseReserve,IBNR,TotalReserve,Check,Difference,Status,PaidCLUltimate,IncurredCLUltimate
0,1988,86.0,86.0,86.000000,0.0,0.000000,0.000000,0.000000,0.0,Reconciles,86.000000,86.000000
1,1989,76.0,76.0,76.000000,0.0,0.000000,0.000000,0.000000,0.0,Reconciles,76.000000,76.000000
2,1990,128.0,128.0,128.000000,0.0,0.000000,0.000000,0.000000,0.0,Reconciles,128.000000,128.000000
3,1991,69.0,69.0,69.000000,0.0,0.000000,0.000000,0.000000,0.0,Reconciles,69.000000,69.000000
4,1992,175.0,197.0,178.750000,22.0,-18.250000,3.750000,3.750000,0.0,Reconciles,178.750000,193.959877
5,1993,24.0,24.0,29.887552,0.0,5.887552,5.887552,5.887552,0.0,Reconciles,29.887552,24.134938
6,1994,143.0,156.0,187.340931,13.0,31.340931,44.340931,44.340931,0.0,Reconciles,187.340931,160.350632
7,1995,26.0,69.0,40.447346,43.0,-28.552654,14.447346,14.447346,0.0,Reconciles,40.447346,76.684681
8,1996,37.0,105.0,99.438423,68.0,-5.561577,62.438423,62.438423,0.0,Reconciles,99.438423,134.582287
9,1997,25.0,134.0,135.086731,109.0,1.086731,110.086731,110.086731,0.0,Reconciles,135.086731,195.115606


So which ultimate is "correct"?

There is no single observed ultimate.

You may have:

| Method      | Ultimate |
| ----------- | -------: |
| Paid CL     |      80M |
| Incurred CL |      78M |
| BF          |      75M |
| Cape Cod    |      77M |


The actuary reviews all indications and selects a reasonable ultimate.

Friedland explicitly notes:

The actuary will ordinarily examine the indications of more than one method when estimating reserves.

This is exactly why.

They first calculate:
Ultimate Loss

and then:
Reserve=Ultimate−Known Amount

where the "known amount" depends on the method:

Paid CL → Ultimate − Paid
Incurred CL → Ultimate − Paid (total reserve)
Incurred CL → Ultimate − Incurred (IBNR)

Current Incurred - Paid = Case Reserve
IBNR = Selected Ultimate − Current Incurred